# Solutions — JSX

Only look here after you've actually tried the exercises in `jsx.ipynb`.

### LESSON 7 — Exercise

`l7h` is redefined here so this notebook runs on its own.

Part 2 is a recursive walk: count this node, then add whatever its element children
contribute. The `typeof child === "object"` test is what keeps the plain strings out.

In [ ]:
// Conceptual model — not React's actual implementation.
function l7h(type, props = {}, ...children) {
  return { type, props: { ...props, children } };
}

// Part 1
const l7today = l7h(
  "section",
  {},
  l7h("h2", {}, "Today"),
  l7h("p", {}, "Fresh bread"),
  l7h("p", {}, "Coffee beans"),
);

console.log(JSON.stringify(l7today, null, 2));

// Part 2
function l7countElements(node) {
  let total = 1;
  for (const child of node.props.children) {
    if (typeof child === "object") total += l7countElements(child);
  }
  return total;
}

console.log(l7countElements(l7today)); // 4

**Common mistake.** Counting the strings too, which gives 7. `"Fresh bread"` is text
inside an element, not an element of its own — and React treats it the same way.

**A neater variant**, if you prefer array methods to loops:

In [ ]:
function l7countElements2(node) {
  return node.props.children
    .filter((child) => typeof child === "object")
    .reduce((sum, child) => sum + l7countElements2(child), 1);
}

console.log(l7countElements2(l7today)); // 4

### LESSON 7 — Mini challenge

**1. The compiled calls.**

```js
jsx(Card, {
  children: jsx("h2", { children: "Revenue" })
});
```

Nesting in JSX becomes nesting in the `children` prop. That is all `children` ever was — a
prop that the JSX syntax fills in for you.

**2. Why `Card` must not be quoted.** `"Card"` would be an HTML tag name, so React would
create a `<card>` element in the DOM and never call your function — exactly the LESSON 4
failure, where the content silently disappeared. The unquoted `Card` is a reference to the
actual function, which is what lets React call it.

**3. What `box` is.** An element: a plain JavaScript object describing a paragraph. Nothing
has appeared on screen. It is a value sitting in a variable, and it will stay that way
until something renders it — which for the top of the tree is the `root.render(...)` call
from LESSON 3.

This is why JSX can go in variables, be returned from functions, or be collected in an
array. It is data until React is told to do something with it.

### LESSON 8 — Exercise

All three are the same move: replace a statement that *assigns* with an expression that
*produces*.

In [ ]:
// 1 — if/else becomes a ternary
const l8score = 72;
const l8status = l8score >= 60 ? "pass" : "fail";
console.log(l8status);

// 2 — the loop becomes join(), which also fixes the trailing separator for free
const l8items = ["bread", "milk", "eggs"];
const l8list = l8items.join(" / ");
console.log(l8list);

// 3 — reduce produces the sum; toFixed(2) formats it
const l8prices = [4.5, 2.25, 9];
const l8total = `${l8prices.reduce((sum, price) => sum + price, 0).toFixed(2)} EUR`;
console.log(l8total);

// None of the three originals could go inside JSX braces: if, for and let are all
// statements. Every rewrite above can, because each one produces a value.

**Common mistakes.**

- Writing `l8items.join(" / ") + " / "` to match the loop's output. The loop's trailing
  separator was a bug, not a requirement — `join` is the reason this rewrite is an
  improvement and not just a translation.
- Reaching for `map` in exercise 2. There is nothing to transform, only to glue. `join`
  alone is enough.
- `toFixed` returns a **string**, not a number. That is fine here, since the answer is a
  string, but adding to it later would concatenate rather than sum.

### LESSON 8 — Mini challenge

| # | fragment | verdict |
|---|---|---|
| 1 | `items.filter((i) => i.done).length` | **expression** — produces a number |
| 2 | `let total = 0` | **statement** — declares; produces nothing |
| 3 | `user ? user.email : "no email"` | **expression** — produces one of two strings |
| 4 | `if (isAdmin) { showPanel() }` | **statement** — runs code; produces nothing |
| 5 | `prices.reduce((a, b) => a + b, 0)` | **expression** — produces a number |
| 6 | `for (const t of tags) { out.push(t) }` | **statement** — loops; produces nothing |

Expressions for the statements:

```js
// 2
0

// 4  — produces a value instead of performing an action
isAdmin ? "admin panel" : ""

// 6
tags.map((t) => t)      // or simply: [...tags]
```

**Why 3 works inside braces and 4 does not.** Both choose between two paths, and that is
not the difference. The difference is what is left behind afterwards.

The ternary in 3 **evaluates to** one of its two branches — the whole fragment *is* a
value, so React can take it and put it on the page. The `if` in 4 evaluates to nothing at
all; it chooses which code to run and then hands back no result. There is nothing for
React to render, so the syntax does not even permit it.

"Produces a value" versus "performs an action" is the whole distinction, and it decides
every time whether something can live inside `{ }`.

### LESSON 9 — Exercise

`l9toCss` and `l9unitless` are repeated here so this notebook runs on its own.

In [ ]:
const l9unitless = new Set(["opacity", "zIndex", "lineHeight", "flex", "fontWeight"]);

function l9toCss(style) {
  return Object.entries(style)
    .map(([key, value]) => {
      const prop = key.replace(/[A-Z]/g, (letter) => "-" + letter.toLowerCase());
      const out = typeof value === "number" && !l9unitless.has(key) ? `${value}px` : value;
      return `${prop}: ${out}`;
    })
    .join("; ");
}

// Part 1 — spread copies, then the later keys win
const l9defaults = { type: "text", required: false, maxLength: 50 };
const l9overrides = { required: true, maxLength: 80 };

const l9attrs = { ...l9defaults, ...l9overrides };
console.log(l9attrs);
console.log(l9defaults); // unchanged — spread built a new object

In [ ]:
// Part 2
function l9barStyle(percent) {
  return {
    width: `${percent}%`,
    backgroundColor: percent >= 70 ? "seagreen" : "goldenrod",
    opacity: percent > 0 ? 1 : 0.4,
  };
}

console.log(l9barStyle(45));
console.log(l9barStyle(90));

console.log(l9toCss(l9barStyle(45)));
console.log(l9toCss(l9barStyle(90)));

**Common mistakes.**

- `Object.assign(l9defaults, l9overrides)` instead of spread. It produces the right
  `l9attrs` but **mutates `l9defaults`**, so the second log shows the overrides too. That
  distinction stops being academic in topic 9, where mutating state is the single most
  common React bug.
- Returning `width: percent` as a number. React would render `45px` — a 45-pixel bar
  regardless of the container. The `%` has to be in the string.
- Putting `opacity` in the unitless set is not optional bookkeeping: without it,
  `opacity: 1` would become `opacity: 1px`, which the browser throws away.

### LESSON 9 — Mini challenge

| # | attribute | verdict |
|---|---|---|
| 1 | `<div class="card">` | **wrong spelling** — React warns *Invalid DOM property `class`. Did you mean `className`?*. The attribute still reaches the DOM and the styling still works, so the warning is the only symptom |
| 2 | `<label for="email">` | **wrong** — same reason; use `htmlFor` |
| 3 | `<input maxlength={10} />` | **wrong** — must be camelCase: `maxLength` |
| 4 | `<div style="color: red">` | **wrong** — `style` takes an object, not a string: `style={{ color: "red" }}` |
| 5 | `<img width="100" alt="" />` | **correct** — a literal string is fine; `width={100}` would be equally fine |

**The two sets of braces in `style={{ fontSize: 16 }}`.** The outer pair is JSX syntax: it
means *the value of this attribute is a JavaScript expression*. The inner pair is an
ordinary object literal. Neither is special to `style`; you are simply passing an object,
and passing any object needs both.

**Why `opacity: 1` and `fontSize: 16` behave differently.** React keeps a list of CSS
properties that take no unit. Anything on that list is written out untouched; everything
else gets `px` appended to a number.

Without the list, React would have only two options. It could append `px` to every number —
breaking `opacity`, `zIndex`, `lineHeight` and `flex`, all of which the browser would then
discard. Or it could append nothing, and make you write `fontSize: "16px"` every single
time. The list buys the convenience of `fontSize: 16` without breaking the unitless
properties, at the cost of one lookup table.

### LESSON 10 — Exercise

Order of checks is the whole exercise: `null` before `typeof`, and `Array.isArray` before
the object test.

In [ ]:
function l10describe(value) {
  if (value === null || value === undefined) return "nothing";
  if (typeof value === "boolean") return "nothing";
  if (Array.isArray(value)) return `list of ${value.length}`;
  if (typeof value === "object") return "error";
  return `text: ${value}`;
}

const l10values = [0, "", "Ada", null, undefined, false, true, [1, 2, 3], { name: "Ada" }];

for (const value of l10values) {
  console.log(l10describe(value));
}

**Common mistakes.**

- `if (!value) return "nothing"` as the first line. It looks tidy and it is wrong twice
  over: `0` and `""` are both falsy, and both are text as far as React is concerned. This
  is the same reflex that causes the `0` bug in topic 7.
- Testing `typeof value === "object"` before `Array.isArray(value)`. Arrays are objects, so
  every array would be reported as an error.
- Forgetting `undefined`. `typeof undefined` is `"undefined"`, not `"object"` — so it falls
  through to the last line and gets described as `text: undefined`, which is exactly the
  bug where a missing value prints the word "undefined" on the page.

### LESSON 10 — Mini challenge

**1. Where a `<div>` breaks the page.** Any time the parent's layout depends on its direct
children.

```jsx
<ul>
  <Items />      {/* if Items returns <div><li/><li/></div> … */}
</ul>
```

Now the `<li>`s are inside a `<div>` inside the `<ul>`, which is invalid HTML and loses the
list styling. The same happens with `display: flex` or `display: grid`: the wrapper becomes
the single flex item and your columns collapse into one. A fragment leaves the children as
direct children, which is what the parent expected.

**2. `// TODO: add avatar` between two tags.** The user reads `// TODO: add avatar` on the
page. Between tags you are in markup, so that is just text — JavaScript comment syntax means
nothing there. It needs to be `{/* TODO: add avatar */}`.

**3. `{[]}` versus `{null}`.** Both put nothing in the page. The difference is why: `null`
is a hole React skips, while `[]` is a list that React renders item by item — and there are
no items. Same visible result, different reason.

This matters later: rendering a list that happens to be empty shows nothing at all, with no
error and no clue. That is why topic 7 asks you to handle the empty case on purpose.

**4. `<p>{user}</p>` with an object.** The author meant `{user.name}` — a value React can
turn into text.

React refuses rather than guessing because there is no correct guess available. Should
`{ name: "Ada", email: "a@b.c" }` become `Ada`? `Ada a@b.c`? `[object Object]`? Any choice
would be wrong for someone, and the last one is what plain JavaScript would do silently —
which is worse than an error, because you would ship it. The error names the problem
immediately:

```text
Objects are not valid as a React child (found: …)
```